# 01 — Dataset Loading
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Load both raw CSV files and understand their structure before any processing.

> The telemetry dataset is in **long format** — one row per measurement.  
> The telecommand dataset logs every ground-station command sent to the spacecraft.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
    'axes.titlesize':12, 'axes.labelsize':10, 'font.size':10
})
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load both CSV files
telemetry_raw   = pd.read_csv('telemetry_train.csv')
telecommand_raw = pd.read_csv('telecommand_train.csv')

print('Telemetry shape  :', telemetry_raw.shape)    # Expected: (10000, 3)
print('Telecommand shape:', telecommand_raw.shape)  # Expected: (161, 3)

### Telemetry Dataset
Three columns: `timestamp` (when measured), `parameter` (what was measured), `value` (reading).

In [ ]:
# Each row = one sensor reading for one parameter at one point in time
display(telemetry_raw.head(10))

In [ ]:
# Note: timestamp is dtype 'object' (string) — needs to be parsed to datetime later
# value is float64 — raw sensor readings
telemetry_raw.info()

In [ ]:
# describe() shows the value column spans a wide range across all 50 parameters
# min, max will look extreme because they come from different physical units
# (e.g., RF signal in -85 dBm vs Solar power in +130 W)
# --> This confirms that SCALING is required before feeding to any ML model
display(telemetry_raw.describe())

In [ ]:
# 50 unique parameters covering Power, Thermal, ADCS, Comms, OBC, Propulsion
print('Unique parameters:', telemetry_raw['parameter'].nunique())
print(sorted(telemetry_raw['parameter'].unique()))

### Telecommand Dataset
Three columns: `timestamp`, `command` (instruction type), `value` (1=success, 0=pending/failed).

In [ ]:
# Each row = one command sent from ground station to the spacecraft
display(telecommand_raw.head(10))

In [ ]:
# value column is integer (0 or 1) — binary execution status
telecommand_raw.info()

In [ ]:
# mean of value ~0.96 means ~96% of commands were executed successfully
# This is expected in normal operations — very few pending/failed commands
display(telecommand_raw.describe())

In [ ]:
# 161 unique command types spanning all spacecraft subsystems
print('Unique commands:', telecommand_raw['command'].nunique())
print(sorted(telecommand_raw['command'].unique()))

---
**Key Takeaways:**
- Telemetry: 10 000 rows, 50 parameters, long format, values in mixed physical units
- Telecommand: 161 rows, 161 unique commands, 96% success rate in normal operations
- `timestamp` is a string in both datasets → must convert to datetime in the next step
- Mixed value ranges in telemetry → scaling mandatory before ML